In [19]:
import numpy as np
import pandas as pd

from SymbolicDSGE import DSGESolver, ModelParser, Shock
from SymbolicDSGE.monte_carlo import (
    MCPipeline,
    reference_filter_step,
    simulation_step,
    wald_test_step,
    raw_data_step,
)

In [ ]:
model, kalman = ModelParser("../../MODELS/POST82.yaml").get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(
    n_state=3,
    n_exog=3,
)

steady_state = np.zeros(5, dtype=np.float64)
reference = solver.solve(compiled, steady_state=steady_state)

dgp_params = {str(k): v for k, v in model.calibration.parameters.items()}

dgp_params["rho_g"] = 0.90
dgp_params["rho_z"] = 0.75

dgp = solver.solve(
    compiled,
    parameters=dgp_params,
    steady_state=steady_state,
)

In [ ]:
T = 200
n_obs = len(reference.compiled.observable_names)

pipeline = MCPipeline(
    [
        simulation_step(
            T=T,
            shocks={
                "g,z": Shock(T=T, dist="norm", multivar=True, seed=0),
                "r": Shock(T=T, dist="norm", seed=1),
            },
            observables=True,
        ),
        reference_filter_step(estimate_R_diag=False),
        wald_test_step(
            "std_innov_mean",
            source="std_innov",
            target=np.zeros(n_obs),
            kind="mean",
            burn_in=20,
        ),
        wald_test_step(
            "std_innov_second_moment",
            source="std_innov",
            target=np.eye(n_obs),
            kind="second_moment",
            burn_in=20,
        ),
    ]
)

In [22]:
mc = pipeline.run(
    reference=reference,
    dgp=dgp,
    n_rep=500,
    retain_payloads=False,
    retain_test_results=False,
)

mc.succeeded, mc.n_successful

(True, 500)

In [25]:
summary = pd.DataFrame(
    {
        name: {
            "mean_statistic": res.mean_statistic,
            "mean_pval": res.mean_pval,
            "rejection_rate": res.rejection_rate,
            "ci_low": res.pval_confidence_interval()[0],
            "ci_high": res.pval_confidence_interval()[1],
        }
        for name, res in mc.summaries.items()
    }
).T
print(summary.round(3))

                         mean_statistic  mean_pval  rejection_rate  ci_low  \
std_innov_mean                    5.652      0.363           0.224   0.190   
std_innov_second_moment        1400.913      0.000           1.000   0.992   

                         ci_high  
std_innov_mean             0.263  
std_innov_second_moment    1.000  
